<a href="https://colab.research.google.com/github/Sushrut0202/CIIMS/blob/main/Automated_NGS_sample_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import os
from google.colab import files

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# how many top isolates per farm-time cluster
TOP_PER_CLUSTER = 3


# ==============================
# LOAD DATA
# ==============================

# Check if INPUT_FILE exists, if not, prompt user to upload
if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Please upload the file.")
    uploaded = files.upload()
    if INPUT_FILE not in uploaded:
        raise FileNotFoundError(f"'{INPUT_FILE}' was not uploaded. Please ensure the correct file is selected.")
    else:
        print(f"'{INPUT_FILE}' uploaded successfully.")

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# use correct MDR count
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "].fillna(0).astype(int)
)

# keep relevant columns
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])

# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# create FARM-TIME cluster (monthly)
df["Cluster_ID"] = (
    df["Farm Name"].astype(str)
    + "_"
    + df["Date of Sample Collection"].dt.to_period("M").astype(str)
)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # pathogen diversity rule
    seen_pathogens = set()

    for _, row in cluster_df.iterrows():

        if len(selected_rows) == TOP_PER_CLUSTER:
            pass

        pathogen = row["Pathogen"]

        # prefer new pathogen types
        if pathogen not in seen_pathogens:
            selected_rows.append(row)
            seen_pathogens.add(pathogen)

        # stop when enough selected
        if len(seen_pathogens) >= TOP_PER_CLUSTER:
            break

# final dataframe
selected_df = pd.DataFrame(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ mNGS selection complete!")
print(f"Output saved: {OUTPUT_FILE}")

'Human Samples.xlsx' not found. Please upload the file.


Saving Human Samples.xlsx to Human Samples.xlsx
'Human Samples.xlsx' uploaded successfully.
✅ mNGS selection complete!
Output saved: mNGS_Human_Selected.xlsx


In [1]:
import pandas as pd
import os
from google.colab import files

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# how many top isolates per farm-time cluster
TOP_PER_CLUSTER = 3


# ==============================
# LOAD DATA
# ==============================

# Check if INPUT_FILE exists, if not, prompt user to upload
if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Please upload the file.")
    uploaded = files.upload()
    if INPUT_FILE not in uploaded:
        raise FileNotFoundError(f"'{INPUT_FILE}' was not uploaded. Please ensure the correct file is selected.")
    else:
        print(f"'{INPUT_FILE}' uploaded successfully.")

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# use correct MDR count
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "].fillna(0).astype(int)
)

# keep relevant columns
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])

# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# create FARM-TIME cluster (monthly)
def make_cluster(row):
    if pd.isna(row["Date of Sample Collection"]):
        return row["Farm Name"]   # farm only
    else:
        return (
            row["Farm Name"]
            + "_"
            + str(row["Date of Sample Collection"].to_period("M"))
        )

df["Cluster_ID"] = df.apply(make_cluster, axis=1)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # pathogen diversity rule
    seen_pathogens = set()

    for _, row in cluster_df.iterrows():

        if len(selected_rows) == TOP_PER_CLUSTER:
            pass

        pathogen = row["Pathogen"]

        # prefer new pathogen types
        if pathogen not in seen_pathogens:
            selected_rows.append(row)
            seen_pathogens.add(pathogen)

        # stop when enough selected
        if len(seen_pathogens) >= TOP_PER_CLUSTER:
            break

# final dataframe
selected_df = pd.DataFrame(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ mNGS selection complete!")
print(f"Output saved: {OUTPUT_FILE}")

'Human Samples.xlsx' not found. Please upload the file.


Saving Human Samples.xlsx to Human Samples.xlsx
'Human Samples.xlsx' uploaded successfully.
✅ mNGS selection complete!
Output saved: mNGS_Human_Selected.xlsx


In [2]:
import pandas as pd
import os
from google.colab import files

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# how many top isolates per farm-time cluster
TOP_PER_CLUSTER = 3


# ==============================
# LOAD DATA
# ==============================

# Check if INPUT_FILE exists, if not, prompt user to upload
if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Please upload the file.")
    uploaded = files.upload()
    if INPUT_FILE not in uploaded:
        raise FileNotFoundError(f"'{INPUT_FILE}' was not uploaded. Please ensure the correct file is selected.")
    else:
        print(f"'{INPUT_FILE}' uploaded successfully.")

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# use correct MDR count
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "].fillna(0).astype(int)
)

# keep relevant columns
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])
# ==============================
# PATHOGEN FILTER
# ==============================

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "s. pneumoniae",
    "p. aeruginosa",
    "a. baumannii"
]

# clean pathogen names
df["Pathogen"] = (
    df["Pathogen"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# keep only target pathogens
df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]
# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# create FARM-TIME cluster (monthly)
def make_cluster(row):
    if pd.isna(row["Date of Sample Collection"]):
        return row["Farm Name"]   # farm only
    else:
        return (
            row["Farm Name"]
            + "_"
            + str(row["Date of Sample Collection"].to_period("M"))
        )

df["Cluster_ID"] = df.apply(make_cluster, axis=1)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # pathogen diversity rule
    seen_pathogens = set()

    for _, row in cluster_df.iterrows():

        if len(selected_rows) == TOP_PER_CLUSTER:
            pass

        pathogen = row["Pathogen"]

        # prefer new pathogen types
        if pathogen not in seen_pathogens:
            selected_rows.append(row)
            seen_pathogens.add(pathogen)

        # stop when enough selected
        if len(seen_pathogens) >= TOP_PER_CLUSTER:
            break

# final dataframe
selected_df = pd.DataFrame(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ mNGS selection complete!")
print(f"Output saved: {OUTPUT_FILE}")

'Human Samples.xlsx' not found. Please upload the file.


Saving Human Samples.xlsx to Human Samples.xlsx
'Human Samples.xlsx' uploaded successfully.
✅ mNGS selection complete!
Output saved: mNGS_Human_Selected.xlsx


In [2]:
import pandas as pd

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected_FINAL.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# target pathogens (lowercase normalized)
TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "s. pneumoniae",
    "p. aeruginosa",
    "a. baumannii"
]

# ==============================
# LOAD DATA
# ==============================

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# clean farm names (IMPORTANT)
df["Farm Name"] = df["Farm Name"].astype(str).str.strip()

# clean pathogen names
df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# keep only pathogens of interest
df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

# correct MDR count (already precomputed)
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "]
    .fillna(0)
    .astype(int)
)

# keep relevant columns only
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])

# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# smart cluster creation
def make_cluster(row):
    if pd.isna(row["Date of Sample Collection"]):
        return row["Farm Name"]
    else:
        return (
            row["Farm Name"]
            + "_"
            + str(row["Date of Sample Collection"].to_period("M"))
        )

df["Cluster_ID"] = df.apply(make_cluster, axis=1)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # keep ONLY highest resistance isolate per pathogen
    best_per_pathogen = (
        cluster_df
        .drop_duplicates(subset="Pathogen", keep="first")
    )

    selected_rows.append(best_per_pathogen)

# combine all clusters
selected_df = pd.concat(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ FINAL mNGS selection created!")
print(f"Output saved to: {OUTPUT_FILE}")

FileNotFoundError: [Errno 2] No such file or directory: 'Human Samples.xlsx'

In [3]:
import os
from google.colab import files

INPUT_FILE = "Human Samples.xlsx"

# Check if INPUT_FILE exists, if not, prompt user to upload
if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Please upload the file.")
    uploaded = files.upload()
    if INPUT_FILE not in uploaded:
        raise FileNotFoundError(f"'{INPUT_FILE}' was not uploaded. Please ensure the correct file is selected.")
    else:
        print(f"'{INPUT_FILE}' uploaded successfully.")

'Human Samples.xlsx' not found. Please upload the file.


Saving Human Samples.xlsx to Human Samples.xlsx
'Human Samples.xlsx' uploaded successfully.


In [4]:
import pandas as pd

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected_FINAL.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# target pathogens (lowercase normalized)
TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "s. pneumoniae",
    "p. aeruginosa",
    "a. baumannii"
]

# ==============================
# LOAD DATA
# ==============================

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# clean farm names (IMPORTANT)
df["Farm Name"] = df["Farm Name"].astype(str).str.strip()

# clean pathogen names
df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# keep only pathogens of interest
df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

# correct MDR count (already precomputed)
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "]
    .fillna(0)
    .astype(int)
)

# keep relevant columns only
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])

# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# smart cluster creation
def make_cluster(row):
    if pd.isna(row["Date of Sample Collection"]):
        return row["Farm Name"]
    else:
        return (
            row["Farm Name"]
            + "_"
            + str(row["Date of Sample Collection"].to_period("M"))
        )

df["Cluster_ID"] = df.apply(make_cluster, axis=1)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # keep ONLY highest resistance isolate per pathogen
    best_per_pathogen = (
        cluster_df
        .drop_duplicates(subset="Pathogen", keep="first")
    )

    selected_rows.append(best_per_pathogen)

# combine all clusters
selected_df = pd.concat(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ FINAL mNGS selection created!")
print(f"Output saved to: {OUTPUT_FILE}")

✅ FINAL mNGS selection created!
Output saved to: mNGS_Human_Selected_FINAL.xlsx


In [5]:
import pandas as pd

# ==============================
# CONFIGURATION
# ==============================

INPUT_FILE = "Human Samples.xlsx"
OUTPUT_FILE = "mNGS_Human_Selected_FINAL.xlsx"

SECTOR = "Human"
SAMPLE_TYPE = "Stool"

# target pathogens (lowercase normalized)
TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "s. pneumoniae",
    "p. aeruginosa",
    "a. baumannii"
]

# ==============================
# LOAD DATA
# ==============================

df = pd.read_excel(INPUT_FILE)

# ==============================
# BASIC CLEANING
# ==============================

# clean farm names (IMPORTANT)
df["Farm Name"] = df["Farm Name"].astype(str).str.strip()

# clean pathogen names
df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# keep only pathogens of interest
df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

# correct MDR count (already precomputed)
df["No_of_ABX_Classes_Resistant"] = (
    df["RESISTANT "]
    .fillna(0)
    .astype(int)
)
# remove invalid rows
df = df[
    (df["No_of_ABX_Classes_Resistant"] > 0) &
    (df["Farm Name"].notna()) &
    (df["Farm Name"] != "nan")
]
# keep relevant columns only
df = df[[
    "Sample Code",
    "Farm Name",
    "Pathogen",
    "Date of Sample Collection",
    "No_of_ABX_Classes_Resistant"
]]

df = df.dropna(subset=["Sample Code", "Pathogen"])

# ==============================
# DATE HANDLING
# ==============================

df["Date of Sample Collection"] = pd.to_datetime(
    df["Date of Sample Collection"],
    errors="coerce"
)

# smart cluster creation
def make_cluster(row):
    if pd.isna(row["Date of Sample Collection"]):
        return row["Farm Name"]
    else:
        return (
            row["Farm Name"]
            + "_"
            + str(row["Date of Sample Collection"].to_period("M"))
        )

df["Cluster_ID"] = df.apply(make_cluster, axis=1)

# ==============================
# SELECTION LOGIC
# ==============================

selected_rows = []

for cluster, cluster_df in df.groupby("Cluster_ID"):

    # sort by resistance (highest first)
    cluster_df = cluster_df.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    # keep ONLY highest resistance isolate per pathogen
    best_per_pathogen = (
        cluster_df
        .drop_duplicates(subset="Pathogen", keep="first")
    )

    selected_rows.append(best_per_pathogen)

# combine all clusters
selected_df = pd.concat(selected_rows)

# add metadata
selected_df["Sector"] = SECTOR
selected_df["Sample_Type"] = SAMPLE_TYPE

# ==============================
# SAVE OUTPUT
# ==============================

selected_df.to_excel(OUTPUT_FILE, index=False)

print("✅ FINAL mNGS selection created!")
print(f"Output saved to: {OUTPUT_FILE}")

✅ FINAL mNGS selection created!
Output saved to: mNGS_Human_Selected_FINAL.xlsx
